[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module3/10-concurrency.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module3/10-concurrency.ipynb)

# Module 3.10 — Concurrency: Threading and Multiprocessing
**Module 3: Automation & Scripting** | Estimated time: 35 minutes

---

## Learning Objectives
By the end of this notebook you will be able to:
- Explain the GIL and why it matters for CPU-bound vs I/O-bound tasks
- Create and manage threads with `threading.Thread` and protect shared state with `Lock`
- Spawn processes with `multiprocessing.Process` and `Pool`
- Use the unified `concurrent.futures` API (same code for threads or processes)
- Pick the right approach for a given workload using benchmarks
- Build practical examples: parallel download simulation and CPU-bound number crunching

In [ ]:
import threading
import multiprocessing
import concurrent.futures
import time
import math
import random
import hashlib
import urllib.request
from pathlib import Path
from collections import defaultdict

print('Python concurrency modules ready.')
print(f'CPU cores available: {multiprocessing.cpu_count()}')

## 1. The GIL — What It Is and Why It Matters

The **Global Interpreter Lock (GIL)** is a mutex inside CPython that ensures only one thread executes Python bytecode at a time.

**Consequences:**
- **I/O-bound tasks** (network requests, disk reads, `time.sleep`): threads work fine because the GIL is released while waiting for I/O
- **CPU-bound tasks** (math, image processing, compression): threads do NOT run in parallel; use `multiprocessing` to bypass the GIL

```
Thread A: ====GIL====|..waiting for network..|====GIL====
Thread B:            |====GIL===============|           
                        ^ GIL released during I/O

Process A: ====CPU====  (separate interpreter, no GIL contention)
Process B:              ====CPU====
```

In [ ]:
# Demonstrate the GIL effect
# CPU-bound work: compute hashes
def cpu_work(n: int) -> int:
    """Hash n random bytes — purely CPU bound."""
    data = random.randbytes(1024)
    for _ in range(n):
        hashlib.sha256(data).hexdigest()
    return n

N_WORK = 5_000

# Sequential
start = time.perf_counter()
cpu_work(N_WORK)
cpu_work(N_WORK)
seq_time = time.perf_counter() - start
print(f'Sequential   : {seq_time:.3f}s')

# Two threads (GIL prevents true parallelism)
start = time.perf_counter()
t1 = threading.Thread(target=cpu_work, args=(N_WORK,))
t2 = threading.Thread(target=cpu_work, args=(N_WORK,))
t1.start(); t2.start()
t1.join();  t2.join()
threaded_time = time.perf_counter() - start
print(f'2 Threads    : {threaded_time:.3f}s  (expected: ~same as sequential)')

# Two processes (bypass the GIL)
start = time.perf_counter()
with multiprocessing.Pool(2) as pool:
    pool.map(cpu_work, [N_WORK, N_WORK])
mp_time = time.perf_counter() - start
print(f'2 Processes  : {mp_time:.3f}s  (expected: ~half of sequential)')

speedup = seq_time / mp_time
print(f'\nMultiprocessing speedup: {speedup:.2f}x')

## 2. `threading.Thread` — Direct Thread Control

In [ ]:
results = {}
results_lock = threading.Lock()

def fetch_page(url: str, delay: float = None) -> None:
    """Simulate an HTTP fetch and store the result."""
    if delay is None:
        delay = random.uniform(0.1, 0.5)  # simulate variable network latency
    time.sleep(delay)
    size = random.randint(5000, 50000)     # fake response size
    with results_lock:  # protect shared dict
        results[url] = {'size': size, 'status': 200}
    print(f'  Fetched {url} ({size} bytes)')


urls = [
    'https://example.com/page/1',
    'https://example.com/page/2',
    'https://example.com/page/3',
    'https://example.com/page/4',
    'https://example.com/page/5',
]

results.clear()
print('Starting 5 threads for parallel page fetch...')
start = time.perf_counter()

threads = [threading.Thread(target=fetch_page, args=(url,), name=f'Fetcher-{i+1}')
           for i, url in enumerate(urls)]
for t in threads:
    t.start()
for t in threads:
    t.join()

elapsed = time.perf_counter() - start
print(f'\nAll done in {elapsed:.3f}s')
print(f'Total bytes: {sum(r["size"] for r in results.values()):,}')

## 3. `threading.Lock` — Protecting Shared State

In [ ]:
# Without lock: race condition
counter_unsafe = 0

def increment_unsafe(n: int):
    global counter_unsafe
    for _ in range(n):
        counter_unsafe += 1  # read-modify-write: not atomic!

threads_unsafe = [threading.Thread(target=increment_unsafe, args=(10_000,)) for _ in range(5)]
for t in threads_unsafe: t.start()
for t in threads_unsafe: t.join()
print(f'Without lock: counter = {counter_unsafe:,}  (expected: 50,000 — likely wrong!)')

# With lock: safe
counter_safe = 0
counter_lock = threading.Lock()

def increment_safe(n: int):
    global counter_safe
    for _ in range(n):
        with counter_lock:
            counter_safe += 1

threads_safe = [threading.Thread(target=increment_safe, args=(10_000,)) for _ in range(5)]
for t in threads_safe: t.start()
for t in threads_safe: t.join()
print(f'With lock   : counter = {counter_safe:,}  (expected: 50,000 — always correct)')

## 4. `multiprocessing.Process` and `Pool`

In [ ]:
# multiprocessing.Pool — parallel map over a list
def is_prime(n: int) -> bool:
    """CPU-bound primality check."""
    if n < 2: return False
    if n == 2: return True
    if n % 2 == 0: return False
    for i in range(3, int(math.isqrt(n)) + 1, 2):
        if n % i == 0:
            return False
    return True

def count_primes_in_range(args):
    start, end = args
    return sum(1 for n in range(start, end) if is_prime(n))


# Split the work into chunks
N = 100_000
chunk_size = N // 4
chunks = [(i * chunk_size, (i + 1) * chunk_size) for i in range(4)]

# Sequential
start = time.perf_counter()
seq_primes = sum(count_primes_in_range(c) for c in chunks)
seq_t = time.perf_counter() - start
print(f'Sequential  : {seq_primes} primes in 0..{N}  ({seq_t:.3f}s)')

# Parallel with Pool
start = time.perf_counter()
with multiprocessing.Pool(processes=4) as pool:
    mp_results = pool.map(count_primes_in_range, chunks)
mp_t = time.perf_counter() - start
print(f'Pool (4)    : {sum(mp_results)} primes  ({mp_t:.3f}s)')
print(f'Speedup     : {seq_t/mp_t:.2f}x')

## 5. `concurrent.futures` — Unified High-Level API

`concurrent.futures` provides `ThreadPoolExecutor` and `ProcessPoolExecutor` with the same interface. You can switch between them by changing one class name.

In [ ]:
def simulate_download(url: str) -> dict:
    """Simulate downloading a URL (I/O-bound)."""
    delay = random.uniform(0.05, 0.3)
    time.sleep(delay)
    size = random.randint(10_000, 500_000)
    return {'url': url, 'bytes': size, 'duration': delay}


download_urls = [f'https://cdn.example.com/file_{i:03d}.dat' for i in range(1, 21)]

# Sequential baseline
start = time.perf_counter()
seq_results = [simulate_download(u) for u in download_urls]
seq_t = time.perf_counter() - start
print(f'Sequential ({len(download_urls)} files): {seq_t:.3f}s')

# ThreadPoolExecutor — best for I/O-bound tasks
start = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
    thread_results = list(executor.map(simulate_download, download_urls))
thread_t = time.perf_counter() - start
print(f'ThreadPool (8 workers): {thread_t:.3f}s  ({seq_t/thread_t:.1f}x faster)')

total_bytes = sum(r['bytes'] for r in thread_results)
print(f'Total downloaded: {total_bytes:,} bytes')

## 6. `as_completed()` — Process Results as They Arrive

In [ ]:
def fetch_with_error(url: str) -> dict:
    """Like simulate_download but occasionally raises an error."""
    time.sleep(random.uniform(0.05, 0.2))
    if 'fail' in url:
        raise ConnectionError(f'Failed to connect to {url}')
    return {'url': url, 'bytes': random.randint(1000, 50000)}


mixed_urls = [
    'https://cdn.example.com/ok_001.dat',
    'https://cdn.fail.example.com/bad.dat',
    'https://cdn.example.com/ok_002.dat',
    'https://cdn.example.com/ok_003.dat',
    'https://cdn.fail.example.com/also_bad.dat',
    'https://cdn.example.com/ok_004.dat',
]

successful = []
failed     = []

with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
    future_to_url = {executor.submit(fetch_with_error, url): url for url in mixed_urls}

    for future in concurrent.futures.as_completed(future_to_url):
        url = future_to_url[future]
        try:
            data = future.result()
            successful.append(data)
            print(f'  OK     {url} ({data["bytes"]:,} bytes)')
        except Exception as exc:
            failed.append({'url': url, 'error': str(exc)})
            print(f'  FAILED {url}: {exc}')

print(f'\nResults: {len(successful)} succeeded, {len(failed)} failed')

## 7. CPU-Bound Benchmark: Threading vs Multiprocessing

In [ ]:
def heavy_cpu(n: int) -> float:
    """Simulate CPU-heavy work: sum of square roots."""
    return sum(math.sqrt(i) for i in range(1, n + 1))

task_sizes = [500_000] * 4  # 4 independent chunks

# Sequential
start = time.perf_counter()
for s in task_sizes: heavy_cpu(s)
seq = time.perf_counter() - start

# ThreadPoolExecutor (GIL prevents true parallelism)
start = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=4) as ex:
    list(ex.map(heavy_cpu, task_sizes))
th = time.perf_counter() - start

# ProcessPoolExecutor (true parallelism)
start = time.perf_counter()
with concurrent.futures.ProcessPoolExecutor(max_workers=4) as ex:
    list(ex.map(heavy_cpu, task_sizes))
mp = time.perf_counter() - start

print('CPU-bound benchmark (4 x 500k sqrt operations):')
print(f'  Sequential         : {seq:.3f}s')
print(f'  ThreadPoolExecutor : {th:.3f}s  ({seq/th:.2f}x)  <- GIL limits improvement')
print(f'  ProcessPoolExecutor: {mp:.3f}s  ({seq/mp:.2f}x)  <- True parallelism')

## 8. Practical: Parallel File Processing

In [ ]:
from pathlib import Path
import hashlib

# Create 20 test files
test_dir = Path('/tmp/pypath_files')
test_dir.mkdir(exist_ok=True)
for i in range(20):
    p = test_dir / f'data_{i:03d}.bin'
    p.write_bytes(random.randbytes(50_000))  # 50 KB each

print(f'Created {len(list(test_dir.iterdir()))} test files')


def hash_file(path: Path) -> dict:
    """Compute SHA-256 hash of a file — I/O + light CPU."""
    h = hashlib.sha256()
    h.update(path.read_bytes())
    return {'file': path.name, 'hash': h.hexdigest(), 'size': path.stat().st_size}


files = list(test_dir.iterdir())

# Sequential
start = time.perf_counter()
seq_hashes = [hash_file(f) for f in files]
seq_t = time.perf_counter() - start
print(f'Sequential : {seq_t:.3f}s')

# ThreadPoolExecutor — good for I/O-bound file work
start = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
    tp_hashes = list(executor.map(hash_file, files))
tp_t = time.perf_counter() - start
print(f'ThreadPool : {tp_t:.3f}s  ({seq_t/tp_t:.1f}x speedup)')

# Verify results match
assert sorted(h['hash'] for h in seq_hashes) == sorted(h['hash'] for h in tp_hashes)
print('Hash results verified: sequential == threaded')

print(f'\nSample hashes:')
for entry in sorted(tp_hashes, key=lambda x: x['file'])[:3]:
    print(f'  {entry["file"]}  {entry["hash"][:16]}...  {entry["size"]:,} bytes')

## 9. Choosing the Right Approach

| Task type | Best tool | Why |
|---|---|---|
| HTTP requests, web scraping | `ThreadPoolExecutor` | I/O-bound; GIL released during network wait |
| File reading/writing | `ThreadPoolExecutor` | I/O-bound |
| Image/video processing | `ProcessPoolExecutor` | CPU-bound; needs true parallelism |
| Data analysis / math | `ProcessPoolExecutor` | CPU-bound |
| Mixed (I/O + light CPU) | `ThreadPoolExecutor` | I/O dominates |
| Async web scraping | `asyncio` + `aiohttp` | Even better for thousands of concurrent requests |
| Background single task | `threading.Thread` | Lightweight, no pool overhead |

**Rule of thumb:** If the task spends most of its time waiting (for network, disk, user input), use threads. If it spends most of its time computing, use processes.

## Practice Exercises

**Exercise 1 — Parallel URL Checker**  
Write a function `check_urls(urls: list[str], workers: int = 10) -> list[dict]` that uses `ThreadPoolExecutor` to send a HEAD request to each URL and returns a list of `{url, status_code, response_time_ms, reachable}` dicts. Test it with 10 public URLs. Handle connection errors gracefully.

**Exercise 2 — Parallel Word Count**  
Given a list of text strings (simulate reading from files), write a function that uses `ProcessPoolExecutor` to count word frequencies in each string in parallel, then merges the results into a single Counter. Compare total execution time against a sequential version for 8 strings of 100,000 words each.

**Exercise 3 — Thread-Safe Cache**  
Implement a `ThreadSafeCache` class with:
- `get(key) -> value | None`
- `set(key, value, ttl_seconds=60)`
- `delete(key)`
- `cleanup()` — remove expired entries

Use `threading.Lock` to make all operations thread-safe. Use `threading.Timer` to call `cleanup()` automatically every 30 seconds. Verify correctness by accessing it from 10 concurrent threads.